In [38]:
!mamba create -n py310 -y
!source /opt/conda/bin/activate py310 && mamba install python=3.10 jupyter mamba -y

!sudo rm /opt/conda/bin/python3
!sudo ln -sf /opt/conda/envs/py310/bin/python3 /opt/conda/bin/python3
!sudo rm /opt/conda/bin/python3.7
!sudo ln -sf /opt/conda/envs/py310/bin/python3 /opt/conda/bin/python3.7
!sudo rm /opt/conda/bin/python
!sudo ln -sf /opt/conda/envs/py310/bin/python3 /opt/conda/bin/python

Traceback (most recent call last):
  File "/opt/conda/bin/mamba", line 7, in <module>
    from mamba.mamba import main
ModuleNotFoundError: No module named 'mamba'
Traceback (most recent call last):
  File "/opt/conda/bin/conda", line 14, in <module>
    from conda.cli import main
ModuleNotFoundError: No module named 'conda'


In [39]:
!python --version

Python 3.10.20


In [40]:
!pip install -r /kaggle/input/datasets/vitli22120188/package/requirements.txt

In [41]:
!pip install opencv-python

In [42]:
!pip list | grep -E "pykan|pennylane"


pennylane                    0.42.3
pennylane_lightning          0.42.0
pykan                        0.2.8


In [43]:
from kan import KAN
import pennylane as qml

In [44]:
# ============================================================
# 0. IMPORTS
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, RobustScaler,MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, balanced_accuracy_score, average_precision_score,
    matthews_corrcoef, cohen_kappa_score, brier_score_loss, roc_auc_score, roc_curve, auc,classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold
import random
import copy

from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.utils import resample
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

import os
import glob
import cv2
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
import torch.nn.functional as F
from sklearn.preprocessing import label_binarize

import numpy as np
import torch
from scipy.spatial.distance import cdist
from scipy.ndimage import binary_erosion

import os
import glob
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import TensorDataset, DataLoader

import torch
import torch.nn as nn
import torch.nn.functional as F


## 1. CONFIGURATION

In [45]:
random_state = 42    
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATASET_ROOT = "/kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"
IMG_SIZE = 128
BATCH_SIZE = 128
N_COMPONENTS = 32  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [46]:
def load_busi_segmentation(dataset_root, img_size=128):
    classes = ["benign", "malignant", "normal"]

    X_images = []
    Y_masks = []
    lesion_flags = []   
    image_paths = []

    for cls in classes:
        cls_dir = os.path.join(dataset_root, cls)
        if not os.path.isdir(cls_dir):
            raise FileNotFoundError(f"Không tìm thấy thư mục: {cls_dir}")

        all_pngs = sorted(glob.glob(os.path.join(cls_dir, "*.png")))

        # chỉ lấy ảnh gốc, bỏ file mask
        image_files = [
            fp for fp in all_pngs
            if "_mask" not in os.path.basename(fp).lower()
        ]

        for img_fp in image_files:
            img_name = os.path.splitext(os.path.basename(img_fp))[0]

            # tìm tất cả mask tương ứng với ảnh này
            # ví dụ:
            # benign (1)_mask.png
            # benign (1)_mask_1.png (nếu có)
            mask_candidates = sorted(
                glob.glob(os.path.join(cls_dir, f"{img_name}_mask*.png"))
            )

            # đọc ảnh grayscale
            img = cv2.imread(img_fp, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"[WARN] Không đọc được ảnh: {img_fp}")
                continue

            img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
            img = img.astype(np.float32) / 255.0

            # tạo mask
            if len(mask_candidates) == 0:
                # normal hoặc ảnh không có mask
                mask = np.zeros((img_size, img_size), dtype=np.float32)
                lesion_flag = 0
            else:
                merged_mask = np.zeros((img_size, img_size), dtype=np.float32)

                for mfp in mask_candidates:
                    m = cv2.imread(mfp, cv2.IMREAD_GRAYSCALE)
                    if m is None:
                        print(f"[WARN] Không đọc được mask: {mfp}")
                        continue

                    m = cv2.resize(m, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
                    m = (m > 0).astype(np.float32)
                    merged_mask = np.maximum(merged_mask, m)

                mask = merged_mask
                lesion_flag = 1 if mask.sum() > 0 else 0

            X_images.append(img)
            Y_masks.append(mask)
            lesion_flags.append(lesion_flag)
            image_paths.append(img_fp)

    X_images = np.array(X_images, dtype=np.float32)
    Y_masks = np.array(Y_masks, dtype=np.float32)
    lesion_flags = np.array(lesion_flags, dtype=np.int64)

    return X_images, Y_masks, lesion_flags, image_paths


X_images, Y_masks, lesion_flags, image_paths = load_busi_segmentation(DATASET_ROOT, IMG_SIZE)

print("Tổng số ảnh:", len(X_images))
print("Shape ảnh:", X_images.shape)
print("Shape mask:", Y_masks.shape)
print("Số ảnh có lesion:", np.sum(lesion_flags == 1))
print("Số ảnh không lesion:", np.sum(lesion_flags == 0))

Tổng số ảnh: 780
Shape ảnh: (780, 128, 128)
Shape mask: (780, 128, 128)
Số ảnh có lesion: 647
Số ảnh không lesion: 133


## 2.Data Loading & Preprocessing

In [47]:
X_train, X_test, y_train_mask, y_test_mask, lesion_train, lesion_test = train_test_split(
    X_images,
    Y_masks,
    lesion_flags,
    test_size=0.2,
    random_state=42,
    stratify=lesion_flags   # stratify theo có/không có lesion
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train lesion:", np.sum(lesion_train == 1))
print("Train normal:", np.sum(lesion_train == 0))
print("Test lesion:", np.sum(lesion_test == 1))
print("Test normal:", np.sum(lesion_test == 0))

Train size: 624
Test size: 156
Train lesion: 518
Train normal: 106
Test lesion: 129
Test normal: 27


In [48]:
X_train_tensor = torch.tensor(X_train[:, None, :, :], dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_mask[:, None, :, :], dtype=torch.float32)

X_test_tensor = torch.tensor(X_test[:, None, :, :], dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_mask[:, None, :, :], dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [49]:
# =========================
# 2. FLATTEN -> TRAIN/VAL/TEST -> SCALE -> PCA
# =========================
# X = X_images

# ### Bỏ Valid 
# X_train, X_val, y_train, y_val = train_test_split(
#     X,y
#     test_size=0.1,   # 10% của 90% => 9% toàn bộ
#     random_state=seed,
#     stratify=y_train_val
# )

# # Bước 1: tách test trước
# X_train_val, X_test, y_train_val, y_test = train_test_split(
#     X, y,
#     test_size=0.1,
#     random_state=seed,
#     stratify=y
# )

# # Bước 2: tách validation từ phần train_val còn lại
# X_train, X_val, y_train, y_val = train_test_split(
#     X_train_val, y_train_val,
#     test_size=0.1,   # 10% của 90% => 9% toàn bộ
#     random_state=seed,
#     stratify=y_train_val
# )

# print("Train size:", len(X_train))
# print("Val size:", len(X_val))
# print("Test size:", len(X_test))
# print("X_images shape:", X_images.shape)
# print("X_train shape:", X_train.shape)

# # =========================
# # SCALE
# # Chỉ fit trên train để tránh data leakage
# # =========================
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_val   = scaler.transform(X_val)
# X_test  = scaler.transform(X_test)

In [50]:
# TRAIN/VAL/TEST
# X_train_tensor = torch.tensor(X_train[:, None, :, :], dtype=torch.float32)
# y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# # X_val_tensor = torch.tensor(X_val[:, None, :, :], dtype=torch.float32)
# # y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# X_test_tensor = torch.tensor(X_test[:, None, :, :], dtype=torch.float32)
# y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# print("X_train_tensor shape:", X_train_tensor.shape)
# # print("X_val_tensor shape:", X_val_tensor.shape)
# print("X_test_tensor shape:", X_test_tensor.shape)

# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# # val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
# test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# # val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# xb, yb = next(iter(train_loader))
# print("batch x shape:", xb.shape)
# print("batch y shape:", yb.shape)

In [51]:
# # =========================
# # PCA
# # Chỉ fit trên train để tránh data leakage
# # =========================
# N_COMPONENTS = 600
# max_pca_components = min(X_train.shape[0], X_train.shape[1])

# if N_COMPONENTS > max_pca_components:
#     print(f"[WARN] N_COMPONENTS={N_COMPONENTS} lớn hơn giới hạn {max_pca_components}. "
#           f"Tự động giảm xuống {max_pca_components}.")
#     N_COMPONENTS = max_pca_components

# pca = PCA(n_components=N_COMPONENTS, random_state=seed)
# X_train = pca.fit_transform(X_train)
# X_val   = pca.transform(X_val)
# X_test  = pca.transform(X_test)

# X_train = X_train.astype(np.float32)
# X_val   = X_val.astype(np.float32)
# X_test  = X_test.astype(np.float32)

# print("X_train shape:", X_train.shape)
# print("X_val shape:", X_val.shape)
# print("X_test shape:", X_test.shape)

# print("Train class 0:", np.sum(y_train == 0))
# print("Train class 1:", np.sum(y_train == 1))
# print("Train class 2:", np.sum(y_train == 2))

# print("Val class 0:", np.sum(y_val == 0))
# print("Val class 1:", np.sum(y_val == 1))
# print("Val class 2:", np.sum(y_val == 2))

# print("Test class 0:", np.sum(y_test == 0))
# print("Test class 1:", np.sum(y_test == 1))
# print("Test class 2:", np.sum(y_test == 2))

## Convert to tensors + π/2 scaling

In [52]:
# X_train = torch.tensor(X_train_final, dtype=torch.float32)
# X_val   = torch.tensor(X_val_final,   dtype=torch.float32)
# X_test  = torch.tensor(X_test_final,  dtype=torch.float32)

# X_train *= torch.cos(X_train)
# X_val   *= torch.cos(X_val)
# X_test  *= torch.cos(X_test)

# psi = 0.5
# X_train *= (psi*np.pi)
# X_val   *= (psi*np.pi)
# X_test  *= (psi*np.pi)


# y_train = torch.tensor(y_train_df.values, dtype=torch.float32).unsqueeze(1)
# y_val   = torch.tensor(y_val_df.values,   dtype=torch.float32).unsqueeze(1)
# y_test  = torch.tensor(y_test_df.values,  dtype=torch.float32).unsqueeze(1)


## 3. Quantum Layer

In [53]:
n_qubits = 1
n_qaoa_layers = 13
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits, seed=seed)

@qml.qnode(dev, interface="torch")
def qnode(inputs,qaoa_weights, weights):
    qml.QAOAEmbedding(inputs, qaoa_weights, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {
    "qaoa_weights": (n_qaoa_layers, 2*n_qubits-1),
    "weights": (n_layers, n_qubits),
}

## 4. LOSS FUNCTION

Một số hàm loss cân nhắc:
* Weighted Cross-Entropy / Weighted BCE
* Cost-sensitive Loss (custom cost)
* Asymmetric Focal Loss
* Class-Balanced Loss (CB Loss)
* Dice Loss / Tversky Loss
* Triplet Loss
* Generalized Cross Entropy (GCE)
* Symmetric Cross Entropy (SCE)
* Focal Loss
* Contrastive Loss
* F1 Loss (surrogate)
* AUC Loss (Pairwise loss)
* LDAM Loss (Label-Distribution-Aware Margin)
* Balanced Softmax Loss
* Equalized Odds / Fairness-aware Loss

### Focal Loss

In [54]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )
        return 1.0 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return self.bce(logits, targets) + self.dice(logits, targets)

In [55]:
class WeightedBCEDiceLoss(nn.Module):
    def __init__(self, pos_weight=1.0, bce_weight=1.0, dice_weight=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], dtype=torch.float32)
        )
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        if self.bce.pos_weight.device != logits.device:
            self.bce.pos_weight = self.bce.pos_weight.to(logits.device)

        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

In [56]:
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        tp = (probs * targets).sum(dim=1)
        fp = (probs * (1 - targets)).sum(dim=1)
        fn = ((1 - probs) * targets).sum(dim=1)

        tversky = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return 1.0 - tversky.mean()

In [57]:
class WeightedBCETverskyLoss(nn.Module):
    def __init__(self, pos_weight=1.0, alpha=0.3, beta=0.7,
                 bce_weight=1.0, tversky_weight=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], dtype=torch.float32)
        )
        self.tversky = TverskyLoss(alpha=alpha, beta=beta)
        self.bce_weight = bce_weight
        self.tversky_weight = tversky_weight

    def forward(self, logits, targets):
        if self.bce.pos_weight.device != logits.device:
            self.bce.pos_weight = self.bce.pos_weight.to(logits.device)

        bce_loss = self.bce(logits, targets)
        tversky_loss = self.tversky(logits, targets)

        return self.bce_weight * bce_loss + self.tversky_weight * tversky_loss

## 5. MODEL ARCHITECTURE

In [58]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

In [59]:
class HybridUNetKANQuantumSegModel(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, kan_in_dim=128):
        super().__init__()

        # ======================
        # U-NET ENCODER
        # ======================
        self.enc1 = DoubleConv(in_channels, 32)     # [B,32,128,128]
        self.pool1 = nn.MaxPool2d(2)                # -> [B,32,64,64]

        self.enc2 = DoubleConv(32, 64)              # [B,64,64,64]
        self.pool2 = nn.MaxPool2d(2)                # -> [B,64,32,32]

        self.enc3 = DoubleConv(64, 128)             # [B,128,32,32]
        self.pool3 = nn.MaxPool2d(2)                # -> [B,128,16,16]

        # bottleneck
        self.bottleneck = DoubleConv(128, 256)      # [B,256,16,16]

        # ======================
        # GLOBAL PROJECTION -> KAN -> QUANTUM
        # ======================
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))   # [B,256,1,1]
        self.pre_kan = nn.Linear(256, kan_in_dim)         # [B,kan_in_dim]
        nn.init.xavier_uniform_(self.pre_kan.weight)
        nn.init.zeros_(self.pre_kan.bias)

        self.kan = KAN(
            width=[kan_in_dim, 100, 50, 25, n_qubits],
            grid=20,
            k=20,
            seed=seed
        )

        self.q_layer = qml.qnn.TorchLayer(qnode, weight_shapes)

        # Quantum output dùng để gate bottleneck feature
        self.quantum_to_bottleneck = nn.Linear(n_qubits, 256)
        nn.init.xavier_uniform_(self.quantum_to_bottleneck.weight)
        nn.init.zeros_(self.quantum_to_bottleneck.bias)

        # ======================
        # U-NET DECODER
        # ======================
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(128 + 128, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(64 + 64, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(32 + 32, 32)

        # output segmentation mask
        self.out_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        # ======================
        # U-NET ENCODER
        # ======================
        e1 = self.enc1(x)                  # [B,32,128,128]
        e2 = self.enc2(self.pool1(e1))     # [B,64,64,64]
        e3 = self.enc3(self.pool2(e2))     # [B,128,32,32]
        b  = self.bottleneck(self.pool3(e3))  # [B,256,16,16]

        # ======================
        # KAN + QUANTUM on bottleneck vector
        # ======================
        z = self.global_pool(b)            # [B,256,1,1]
        z = torch.flatten(z, 1)            # [B,256]
        z = self.pre_kan(z)                # [B,kan_in_dim]

        z = self.kan(z)                    # [B,n_qubits]
        z = self.q_layer(z)                # [B,n_qubits]

        # map quantum vector -> channel attention for bottleneck
        gate = self.quantum_to_bottleneck(z)   # [B,256]
        gate = torch.sigmoid(gate).unsqueeze(-1).unsqueeze(-1)  # [B,256,1,1]

        # apply gating lên bottleneck
        b = b * gate

        # ======================
        # U-NET DECODER
        # ======================
        d3 = self.up3(b)                   # [B,128,32,32]
        d3 = torch.cat([d3, e3], dim=1)    # [B,256,32,32]
        d3 = self.dec3(d3)                 # [B,128,32,32]

        d2 = self.up2(d3)                  # [B,64,64,64]
        d2 = torch.cat([d2, e2], dim=1)    # [B,128,64,64]
        d2 = self.dec2(d2)                 # [B,64,64,64]

        d1 = self.up1(d2)                  # [B,32,128,128]
        d1 = torch.cat([d1, e1], dim=1)    # [B,64,128,128]
        d1 = self.dec1(d1)                 # [B,32,128,128]

        out = self.out_conv(d1)            # [B,1,128,128] logits
        return out

## 6. Class weight smoothing 

## 7. TRAINING SETUP

In [60]:
y = np.array(y_train_mask)
n1 = y.sum()
n0 = y.size - n1
print(n1) 
print(n0) 

778919.0
9444697.0


In [61]:
#GPU 
def compute_manual_pos_weight(y_train_mask, lesion_scale=5.0, smooth=1e-6):
    """
    y_train_mask: numpy array shape [N,H,W] hoặc [N,1,H,W], giá trị 0/1
    lesion_scale: hệ số tay bạn muốn đặt cho lớp 1, ví dụ 5.0
    """
    y = np.array(y_train_mask)

    n1 = y.sum()
    n0 = y.size - n1
    smooth = 0 
    w1 = lesion_scale / (n1 + smooth)
    w0 = lesion_scale*20 / (n0 + smooth)

    pos_weight = w1 / (w0 + smooth)

    print(f"n1 (positive pixels) = {n1}")
    print(f"n0 (negative pixels) = {n0}")
    print(f"w1 = {w1:.12f}")
    print(f"w0 = {w0:.12f}")
    print(f"pos_weight = {pos_weight:.6f}")

    return float(pos_weight)
pos_weight = compute_manual_pos_weight(y_train_mask, lesion_scale=5.0)
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )
        return 1.0 - dice.mean()


class WeightedBCEDiceLoss(nn.Module):
    def __init__(self, pos_weight, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.register_buffer("pos_weight_tensor", torch.tensor(pos_weight, dtype=torch.float32))
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.pos_weight_tensor
        )
        dice = self.dice(logits, targets)
        return self.bce_weight * bce + self.dice_weight * dice
pos_weight = compute_manual_pos_weight(y_train_mask, lesion_scale=5.0)

criterion = WeightedBCEDiceLoss(
    pos_weight=pos_weight,
    bce_weight=1.0,
    dice_weight=1.0
)
criterion = criterion.to(device)

n1 (positive pixels) = 778919.0
n0 (negative pixels) = 9444697.0
w1 = 0.000006419153
w0 = 0.000010587953
pos_weight = 0.606269
n1 (positive pixels) = 778919.0
n0 (negative pixels) = 9444697.0
w1 = 0.000006419153
w0 = 0.000010587953
pos_weight = 0.606269


In [62]:
# def compute_manual_pos_weight(y_train_mask, lesion_scale=5.0, smooth=1e-6):
#     """
#     y_train_mask: numpy array shape [N,H,W] hoặc [N,1,H,W], giá trị 0/1
#     lesion_scale: hệ số tay bạn muốn đặt cho lớp 1, ví dụ 5.0
#     """
#     y = np.array(y_train_mask)

#     n1 = y.sum()
#     n0 = y.size - n1

#     w1 = lesion_scale / (n1 + smooth)
#     w0 = lesion_scale*5 / (n0 + smooth)

#     pos_weight = w1 / (w0 + smooth)

#     print(f"n1 (positive pixels) = {n1}")
#     print(f"n0 (negative pixels) = {n0}")
#     print(f"w1 = {w1:.12f}")
#     print(f"w0 = {w0:.12f}")
#     print(f"pos_weight = {pos_weight:.6f}")

#     return float(pos_weight)
# pos_weight = compute_manual_pos_weight(y_train_mask, lesion_scale=5.0)
# class DiceLoss(nn.Module):
#     def __init__(self, smooth=1e-6):
#         super().__init__()
#         self.smooth = smooth

#     def forward(self, logits, targets):
#         probs = torch.sigmoid(logits)
#         probs = probs.view(probs.size(0), -1)
#         targets = targets.view(targets.size(0), -1)

#         intersection = (probs * targets).sum(dim=1)
#         dice = (2.0 * intersection + self.smooth) / (
#             probs.sum(dim=1) + targets.sum(dim=1) + self.smooth
#         )
#         return 1.0 - dice.mean()


# class WeightedBCEDiceLoss(nn.Module):
#     def __init__(self, pos_weight=1.0, bce_weight=1.0, dice_weight=1.0):
#         super().__init__()
#         self.register_buffer(
#             "pos_weight_tensor",
#             torch.tensor([pos_weight], dtype=torch.float32)
#         )
#         self.bce_weight = bce_weight
#         self.dice_weight = dice_weight
#         self.dice = DiceLoss()

#     def forward(self, logits, targets):
#         bce = F.binary_cross_entropy_with_logits(
#             logits,
#             targets,
#             pos_weight=self.pos_weight_tensor
#         )
#         dice = self.dice(logits, targets)
#         return self.bce_weight * bce + self.dice_weight * dice
# pos_weight = compute_manual_pos_weight(y_train_mask, lesion_scale=5.0)

# criterion = WeightedBCEDiceLoss(
#     pos_weight=pos_weight,
#     bce_weight=1.0,
#     dice_weight=1.0
# )

In [63]:
# def compute_image_level_manual_weight(lesion_flags, lesion_scale=5.0, smooth=1e-6):
#     lesion_flags = np.array(lesion_flags)
#     n1 = np.sum(lesion_flags == 1)   # số ảnh có lesion
#     n0 = np.sum(lesion_flags == 0)   # số ảnh normal

#     w1 = lesion_scale / (n1 + smooth)
#     w0 = 1.0 / (n0 + smooth)

#     pos_weight = w1 / (w0 + smooth)

#     print(f"n1_images = {n1}")
#     print(f"n0_images = {n0}")
#     print(f"w1 = {w1:.12f}")
#     print(f"w0 = {w0:.12f}")
#     print(f"pos_weight = {pos_weight:.6f}")

#     return float(pos_weight)

# pos_weight = compute_manual_pos_weight(y_train_mask, lesion_scale=5.0)
# criterion = WeightedBCEDiceLoss(
#     pos_weight=pos_weight,
#     bce_weight=1.0,
#     dice_weight=1.0
# )

In [64]:
# def compute_soft_pos_weight(Y_masks, method="sqrt", clip_min=1.0, clip_max=20.0):
#     """
#     Y_masks: numpy array [N, H, W], giá trị 0/1
#     """
#     pos_pixels = Y_masks.sum()
#     total_pixels = Y_masks.size
#     neg_pixels = total_pixels - pos_pixels

#     raw_ratio = neg_pixels / (pos_pixels + 1e-6)

#     if method == "sqrt":
#         soft_weight = np.sqrt(raw_ratio)
#     elif method == "log":
#         soft_weight = np.log1p(raw_ratio)
#     elif method == "raw":
#         soft_weight = raw_ratio
#     else:
#         raise ValueError("method must be one of: 'sqrt', 'log', 'raw'")

#     soft_weight = float(np.clip(soft_weight, clip_min, clip_max))

#     print(f"pos_pixels={pos_pixels}")
#     print(f"neg_pixels={neg_pixels}")
#     print(f"raw_ratio={raw_ratio:.4f}")
#     print(f"soft_pos_weight={soft_weight:.4f}")

#     return soft_weight
# pos_weight = compute_soft_pos_weight(Y_masks=y_train_mask, method="sqrt", clip_max=15.0)

# criterion = WeightedBCEDiceLoss(
#     pos_weight=pos_weight,
#     bce_weight=1.0,
#     dice_weight=1.0
# )

In [65]:
# pos_weight = compute_soft_pos_weight(y_train_mask, method="sqrt", clip_max=12.0)

# criterion = WeightedBCETverskyLoss(
#     pos_weight=pos_weight,
#     alpha=0.3,
#     beta=0.7,
#     bce_weight=1.0,
#     tversky_weight=1.0
# )

In [66]:
# # criterion = BCEDiceLoss()
# criterion = WeightedBCEDiceLoss(
#     pos_weight=8.0,   # tăng lớp 1 mạnh hơn lớp 0
#     bce_weight=1.0,
#     dice_weight=1.0
# )

In [67]:
# Dùng 2 GPU 
# if torch.cuda.device_count() > 1:
#     print("Using", torch.cuda.device_count(), "GPUs")
#     model = nn.DataParallel(model)

# model = model.to(device)

In [68]:
model = HybridUNetKANQuantumSegModel(
    in_channels=1,
    out_channels=1,
    kan_in_dim=128
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Early stopping
patience = 1000
best_loss = np.inf
counter = 0
best_state = None


checkpoint directory created: ./model
saving model version 0.0


## 8. TRAINING LOOP

### Train 

In [69]:
# checkpoint = {
#         "epoch": epoch + 1,
#         "model_state_dict": model.state_dict(),
#         "optimizer_state_dict": optimizer.state_dict(),
#         "loss": train_loss,
#     }
# torch.save(checkpoint, f"checkpoint_epoch_{epoch+1}.pth")

In [70]:
# checkpoint = torch.load("/kaggle/input/models/nhatlong1103/checkpoint-113/pytorch/default/1/checkpoint_epoch_113.pth")
# model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]

## Epochs 

In [ ]:
num_more_epochs = 100
for epoch in range(start_epoch, start_epoch + num_more_epochs):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device).float()

        optimizer.zero_grad()
        out = model(x)              # [B,1,H,W]
        loss = criterion(out, y)    # scalar

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}")
    # ===== SAVE CHECKPOINT =====
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": train_loss,
    }

    torch.save(checkpoint, f"checkpoint_epoch_{epoch+1}.pth")

Epoch 214: train_loss=1.4790
Epoch 215: train_loss=1.3544
Epoch 216: train_loss=1.2849
Epoch 217: train_loss=1.2319
Epoch 218: train_loss=1.2011
Epoch 219: train_loss=1.1711
Epoch 220: train_loss=1.1423
Epoch 221: train_loss=1.1192
Epoch 222: train_loss=1.0963
Epoch 223: train_loss=1.0679
Epoch 224: train_loss=1.0470
Epoch 225: train_loss=1.0278
Epoch 226: train_loss=1.0052
Epoch 227: train_loss=0.9810
Epoch 228: train_loss=0.9620
Epoch 229: train_loss=0.9372
Epoch 230: train_loss=0.9177
Epoch 231: train_loss=0.9097
Epoch 232: train_loss=0.9112
Epoch 233: train_loss=0.8814
Epoch 234: train_loss=0.8590
Epoch 235: train_loss=0.8424
Epoch 236: train_loss=0.8200
Epoch 237: train_loss=0.7955
Epoch 238: train_loss=0.7721
Epoch 239: train_loss=0.7540
Epoch 240: train_loss=0.7406
Epoch 241: train_loss=0.7480
Epoch 242: train_loss=0.7129
Epoch 243: train_loss=0.6968
Epoch 244: train_loss=0.6804
Epoch 245: train_loss=0.6709
Epoch 246: train_loss=0.6529
Epoch 247: train_loss=0.6303
Epoch 248: tra

In [ ]:
def dice_score(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    dice = (2 * intersection + smooth) / (preds.sum(dim=1) + targets.sum(dim=1) + smooth)
    return dice.mean().item()


def iou_score(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou.mean().item()


def precision_recall_specificity(preds, targets, threshold=0.5, smooth=1e-6):
    preds = (preds > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    TP = (preds * targets).sum(dim=1)
    FP = (preds * (1 - targets)).sum(dim=1)
    FN = ((1 - preds) * targets).sum(dim=1)
    TN = ((1 - preds) * (1 - targets)).sum(dim=1)

    precision = (TP + smooth) / (TP + FP + smooth)
    recall = (TP + smooth) / (TP + FN + smooth)
    specificity = (TN + smooth) / (TN + FP + smooth)

    return (
        precision.mean().item(),
        recall.mean().item(),
        specificity.mean().item()
    )


def brier_score_multidim(probs, targets):
    # probs, targets: [B,1,H,W]
    return torch.mean((probs - targets) ** 2).item()


# =========================
# SURFACE / DISTANCE METRICS
# =========================
def get_surface_points(mask):
    """
    mask: numpy array 2D, binary {0,1}
    return: coordinates of boundary pixels, shape [N, 2]
    """
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return np.empty((0, 2), dtype=np.int32)

    eroded = binary_erosion(mask)
    surface = mask ^ eroded
    points = np.argwhere(surface)
    return points


def hausdorff_distance_fallback(pred_mask, true_mask):
    """
    pred_mask, true_mask: numpy 2D binary arrays
    """
    pred_pts = get_surface_points(pred_mask)
    true_pts = get_surface_points(true_mask)

    # Handle empty-mask cases
    if len(pred_pts) == 0 and len(true_pts) == 0:
        return 0.0
    if len(pred_pts) == 0 or len(true_pts) == 0:
        return np.nan

    dist_matrix = cdist(pred_pts, true_pts, metric="euclidean")
    hd_forward = dist_matrix.min(axis=1).max()
    hd_backward = dist_matrix.min(axis=0).max()
    return max(hd_forward, hd_backward)


def average_surface_distance_fallback(pred_mask, true_mask):
    """
    Symmetric ASD
    """
    pred_pts = get_surface_points(pred_mask)
    true_pts = get_surface_points(true_mask)

    if len(pred_pts) == 0 and len(true_pts) == 0:
        return 0.0
    if len(pred_pts) == 0 or len(true_pts) == 0:
        return np.nan

    dist_matrix = cdist(pred_pts, true_pts, metric="euclidean")
    asd_forward = dist_matrix.min(axis=1).mean()
    asd_backward = dist_matrix.min(axis=0).mean()
    return (asd_forward + asd_backward) / 2.0


# =========================
# OPTIONAL: USE LIBRARY IF AVAILABLE
# =========================
try:
    from medpy.metric.binary import hd as medpy_hd
    from medpy.metric.binary import asd as medpy_asd
    MEDPY_AVAILABLE = True
except Exception:
    MEDPY_AVAILABLE = False


def hausdorff_distance(pred_mask, true_mask):
    pred_mask = pred_mask.astype(np.uint8)
    true_mask = true_mask.astype(np.uint8)

    if pred_mask.sum() == 0 and true_mask.sum() == 0:
        return 0.0
    if pred_mask.sum() == 0 or true_mask.sum() == 0:
        return np.nan

    if MEDPY_AVAILABLE:
        try:
            return float(medpy_hd(pred_mask, true_mask))
        except Exception:
            pass

    return hausdorff_distance_fallback(pred_mask, true_mask)


def average_surface_distance(pred_mask, true_mask):
    pred_mask = pred_mask.astype(np.uint8)
    true_mask = true_mask.astype(np.uint8)

    if pred_mask.sum() == 0 and true_mask.sum() == 0:
        return 0.0
    if pred_mask.sum() == 0 or true_mask.sum() == 0:
        return np.nan

    if MEDPY_AVAILABLE:
        try:
            return float(medpy_asd(pred_mask, true_mask))
        except Exception:
            pass

    return average_surface_distance_fallback(pred_mask, true_mask)

## 9. EVALUATION

In [ ]:
model.eval()
test_loss = 0.0

dice_list = []
iou_list = []
precision_list = []
recall_list = []
specificity_list = []
brier_list = []
hd_list = []
asd_list = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device).float()          # [B,1,H,W]

        out = model(x)                    # [B,1,H,W]
        loss = criterion(out, y)
        probs = torch.sigmoid(out)        # [B,1,H,W]

        test_loss += loss.item() * x.size(0)

        # batch-level metrics
        dice_list.append(dice_score(probs, y))
        iou_list.append(iou_score(probs, y))

        p, r, s = precision_recall_specificity(probs, y)
        precision_list.append(p)
        recall_list.append(r)
        specificity_list.append(s)

        brier_list.append(brier_score_multidim(probs, y))

        # sample-wise HD / ASD
        preds_bin = (probs > 0.5).float().cpu().numpy()
        y_true_np = y.cpu().numpy()

        for i in range(preds_bin.shape[0]):
            pred_mask = preds_bin[i, 0]
            true_mask = y_true_np[i, 0]

            hd_val = hausdorff_distance(pred_mask, true_mask)
            asd_val = average_surface_distance(pred_mask, true_mask)

            if not np.isnan(hd_val):
                hd_list.append(hd_val)
            if not np.isnan(asd_val):
                asd_list.append(asd_val)
test_loss /= len(test_loader.dataset)

## Metrics 

In [ ]:
print(f"Test loss: {test_loss:.4f}")
print(f"Mean Dice (F1): {np.mean(dice_list):.4f}")
print(f"Mean IoU (Jaccard): {np.mean(iou_list):.4f}")
print(f"Mean Precision: {np.mean(precision_list):.4f}")
print(f"Mean Recall (Sensitivity): {np.mean(recall_list):.4f}")
print(f"Mean Specificity: {np.mean(specificity_list):.4f}")
print(f"Mean Brier Score: {np.mean(brier_list):.4f}")